# Suzuki Yield Prediction — Feature Engineering Report

This notebook documents the end-to-end feature engineering pipeline for predicting the UV yield of Suzuki–Miyaura cross-coupling reactions.

**Input:** `data_suzuki_cleaned.csv` — 5760 reactions × 8 columns  
**Outputs:** `.npy` arrays ready for model training

---

## 1. Dataset Overview

Each row in the dataset is one Suzuki reaction described by:

| Column | Type | Description |
|---|---|---|
| `reactant_1` | string | Quinoline coupling partner (halide, triflate, boronic acid…) |
| `reactant_2` | string | Pyrazole coupling partner (boronic acid, ester, trifluoroborate, bromide) |
| `ligand` | string / NaN | Phosphine ligand used (NaN = no ligand) |
| `ligand_eq` | float | Equivalents of ligand |
| `base` | string / NaN | Base used (NaN = no base) |
| `base_eq` | float | Equivalents of base |
| `solvent` | string | Reaction solvent |
| `yield_uv` | float | **Target**: UV-measured yield (%) |

In [ ]:
import pandas as pd
df = pd.read_csv('data_suzuki_cleaned.csv')
print(df.shape)
df.head()

In [ ]:
# Unique values per condition column
for col in ['reactant_1', 'reactant_2', 'ligand', 'base', 'solvent']:
    vals = df[col].dropna().unique()
    print(f"{col} ({len(vals)}): {vals}")

---
## 2. Feature Engineering Strategy

The feature vector for each reaction is built by concatenating two types of representations:

```
[ DRFP (2048 bits) | RDKit ligand (217) | RDKit base (217) | RDKit solvent (217) ]
                                                             → 2699 raw features
```

### 2a. DRFP — Differential Reaction Fingerprint

DRFP encodes the **chemical transformation** (reactants → products) as a binary vector using circular substructures (Morgan-like) computed on the difference between product and reactant atom environments.

Key parameters:
- `n_folded_length=2048` — output vector size  
- `radius=3` — max circular substructure radius  
- `min_radius=0` — includes individual atoms  
- `rings=True` — whole rings are added as substructures  

Since no product structure is known, the reaction SMILES uses an empty product side:  
`reactant1.reactant2>>`

> **Why DRFP for reactants only?**  
> The reactants define the substrate scope. The chemical transformation encoded by DRFP captures which bonds are present (and absent) across the two coupling partners — a signal directly tied to reaction success.

### 2b. RDKit Descriptors — Conditions (Ligand / Base / Solvent)

The 217 standard RDKit molecular descriptors (MW, logP, TPSA, ring counts, partial charges, BCUT2D, etc.) are computed for each condition molecule.

**Encoding choices:**
- Absent ligand / base (`NaN` in CSV) → **zero vector** (not imputation, deliberate absence)
- Inorganic salts → encoded by their **active species** (e.g. K₃PO₄ → H₃PO₄ as neutral proxy)
- Mixed solvent (MeOH/H₂O 9:1) → **majority solvent** (MeOH); weighted average not implemented
- Fe-containing ligands (dtbpf, dppf) → simplified SMILES; ~12 descriptors return NaN → dropped in Step 5

---
## 3. SMILES Mapping

String names in the CSV are mapped to canonical SMILES strings before any fingerprint or descriptor computation.

In [ ]:
REACTANT_1_SMILES = {
    '6-chloroquinoline':                     'Clc1ccc2ncccc2c1',
    '6-Bromoquinoline':                      'Brc1ccc2ncccc2c1',
    '6-triflatequinoline':                   'FC(F)(F)S(=O)(=O)Oc1ccc2ncccc2c1',
    '6-Iodoquinoline':                       'Ic1ccc2ncccc2c1',
    '6-quinoline-boronic acid hydrochloride': '[Cl-].OB(O)c1ccc2[nH+]cccc2c1',
    'Potassium quinoline-6-trifluoroborate':  '[K+].F[B-](F)(F)c1ccc2ncccc2c1',
    '6-Quinolineboronic acid pinacol ester':  'CC1(C)OB(c2cc3cccnc3cc2)OC1(C)C',
}
REACTANT_2_SMILES = {
    '2a, Boronic Acid':    'Cc1ccc2c(cnn2C2CCCCO2)c1B(O)O',
    '2b, Boronic Ester':   'Cc1ccc2c(cnn2C2CCCCO2)c1B1OC(C)(C)C(C)(C)O1',
    '2c, Trifluoroborate': 'Cc1ccc2c(cnn2C2CCCCO2)c1[B-](F)(F)F.[K+]',
    '2d, Bromide':         'Cc1ccc2c(cnn2C2CCCCO2)c1Br',
}
print("Reactant 1 SMILES:", list(REACTANT_1_SMILES.keys()))
print("Reactant 2 SMILES:", list(REACTANT_2_SMILES.keys()))

---
## 4. DRFP Computation

In [ ]:
import numpy as np
from drfp import DrfpEncoder

reaction_smiles = (
    df['reactant_1'].map(REACTANT_1_SMILES) + '.' +
    df['reactant_2'].map(REACTANT_2_SMILES) + '>>'
).tolist()

print('Sample reaction SMILES:', reaction_smiles[0])

In [ ]:
fps = DrfpEncoder.encode(
    reaction_smiles,
    n_folded_length=2048,
    radius=3, min_radius=0, rings=True,
    show_progress_bar=True
)
drfp_array = np.array(fps)
print('DRFP shape:', drfp_array.shape)
print('All-zero rows (invalid SMILES):', (drfp_array.sum(axis=1) == 0).sum())
print(f'Mean bit density: {drfp_array.mean():.4f}')

---
## 5. RDKit Descriptor Computation

In [ ]:
import math
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

LIGAND_SMILES = {
    'P(tBu)3': 'CC(C)(C)P(C(C)(C)C)C(C)(C)C',
    'P(Ph)3 ': 'P(c1ccccc1)(c1ccccc1)c1ccccc1',
    'AmPhos':  'CN(C)c1ccc(P(C(C)(C)C)C(C)(C)C)cc1',
    'P(Cy)3':  'P(C1CCCCC1)(C1CCCCC1)C1CCCCC1',
    'P(o-Tol)3': 'P(c1ccccc1C)(c1ccccc1C)c1ccccc1C',
    'CataCXium A': 'CCCCP(C12CC3CC(CC(C3)C1)C2)C12CC3CC(CC(C3)C1)C2',
    'SPhos':   'COc1cccc(OC)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1',
    'dtbpf':   'CC(C)(C)P([Fe]P(C(C)(C)C)C(C)(C)C)C(C)(C)C',
    'XPhos':   'CC(C)c1cc(C(C)C)cc(C(C)C)c1-c1ccccc1P(C1CCCCC1)C1CCCCC1',
    'dppf':    'O=P(c1ccccc1)(c1ccccc1)[Fe]P(=O)(c1ccccc1)c1ccccc1',
    'Xantphos': 'O=P(c1ccccc1)(c1ccccc1)c1c(oc2c(C(C)(C)C3)cccc12)cccc3P(=O)(c1ccccc1)c1ccccc1',
}
BASE_SMILES = {
    'NaOH': 'O', 'NaHCO3': 'OC(=O)O', 'CsF': 'F',
    'K3PO4': 'OP(=O)(O)O', 'KOH': 'O', 'LiOtBu': 'OC(C)(C)C', 'Et3N': 'CCN(CC)CC',
}
SOLVENT_SMILES = {
    'MeCN': 'CC#N', 'THF': 'C1CCOC1', 'DMF': 'CN(C)C=O',
    'MeOH': 'OC', 'MeOH/H2O_V2 9:1': 'OC', 'THF_V2': 'C1CCOC1',
}

DESC_NAMES = [d[0] for d in Descriptors.descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(DESC_NAMES)
print(f'{len(DESC_NAMES)} RDKit descriptors')

def smiles_to_descriptors(smi):
    parts = smi.split('.')
    best = max(parts, key=lambda s: (Chem.MolFromSmiles(s) or Chem.MolFromSmiles('C')).GetNumAtoms())
    mol = Chem.MolFromSmiles(best)
    return np.array(calculator.CalcDescriptors(mol), dtype=float) if mol else np.full(len(DESC_NAMES), np.nan)

ligand_cache  = {k: smiles_to_descriptors(v) for k, v in LIGAND_SMILES.items()}
base_cache    = {k: smiles_to_descriptors(v) for k, v in BASE_SMILES.items()}
solvent_cache = {k: smiles_to_descriptors(v) for k, v in SOLVENT_SMILES.items()}
ZERO = np.zeros(len(DESC_NAMES))

def get_desc(cache, key):
    return ZERO if isinstance(key, float) and math.isnan(key) else cache[key]

ligand_array  = np.array([get_desc(ligand_cache,  l) for l in df['ligand']])
base_array    = np.array([get_desc(base_cache,    b) for b in df['base']])
solvent_array = np.array([get_desc(solvent_cache, s) for s in df['solvent']])

print(f'Ligand: {ligand_array.shape}, Base: {base_array.shape}, Solvent: {solvent_array.shape}')

---
## 6. Concatenation & Cleaning

The four arrays are concatenated into a single feature matrix, then columns are removed if they contain:
- **NaN**: caused by Fe-containing ligands (dtbpf, dppf) — ~12 BCUT/partial-charge descriptors
- **Inf**: numerical overflow in certain RDKit descriptors
- **Zero variance**: constant across all reactions → useless for ML

In [ ]:
X_raw = np.concatenate([drfp_array, ligand_array, base_array, solvent_array], axis=1)
print(f'Raw shape: {X_raw.shape}')  # expected (5760, 2699)

nan_mask = np.isnan(X_raw).any(0)
inf_mask = np.isinf(X_raw).any(0)
var_mask = np.var(X_raw, 0) == 0
drop     = nan_mask | inf_mask | var_mask
X        = X_raw[:, ~drop]

print(f'Dropped: {drop.sum()} (NaN={nan_mask.sum()}, Inf={inf_mask.sum()}, Const={var_mask.sum()})')
print(f'Final shape: {X.shape}')

---
## 7. Target Distribution

In [ ]:
import matplotlib.pyplot as plt

y = df['yield_uv'].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y, bins=50, color='steelblue', edgecolor='white')
axes[0].set(title='Yield distribution', xlabel='yield_uv (%)', ylabel='Count')
axes[1].boxplot(y, vert=True)
axes[1].set(title='Yield boxplot', ylabel='yield_uv (%)')
plt.tight_layout()
plt.show()

print(f'Min: {y.min():.2f}  |  Max: {y.max():.2f}  |  Mean: {y.mean():.2f}  |  Std: {y.std():.2f}')

---
## 8. Train / Test Split & Save

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}   |  y_test: {y_test.shape}')

for name, arr in [('X_features', X), ('y_target', y),
                   ('X_train', X_train), ('X_test', X_test),
                   ('y_train', y_train), ('y_test', y_test)]:
    np.save(f'{name}.npy', arr)

print('Saved ✅')

---
## 9. Summary

| Step | Output | Shape |
|---|---|---|
| DRFP (reactants 1 & 2) | `drfp_array` | (N, 2048) |
| RDKit ligand descriptors | `ligand_array` | (N, 217) |
| RDKit base descriptors | `base_array` | (N, 217) |
| RDKit solvent descriptors | `solvent_array` | (N, 217) |
| Raw concatenation | `X_raw` | (N, 2699) |
| After cleaning | `X` | (N, ~2680) |

**Design decisions worth noting:**
- DRFP encodes substrate scope; RDKit descriptors encode reaction conditions.
- Absent ligand/base → zero vector, not mean imputation. This preserves the semantic difference between "no ligand" and "an average ligand".
- Inorganic salt proxies (e.g., K₃PO₄ → H₃PO₄) are a compromise. A more principled approach could use Steinmetz-style ionic descriptors.
- Mixed solvent (MeOH/H₂O 9:1) could be improved with a weighted descriptor average.

**Saved files for modelling:**
```
X_features.npy   — full feature matrix
y_target.npy     — full target vector
X_train.npy      — 80% training features
X_test.npy       — 20% test features
y_train.npy      — training targets
y_test.npy       — test targets
```